# Colab T4 VGGT Runner

Use this notebook for the repo-owned `cloud_vggt_job.zip` package generated by `fpv vggt cloud-job`.

Set `Runtime > Change runtime type > T4 GPU` before running. This job is for offline historical-media reconstruction only: no geolocation, no meters, relative VGGT frame, local-only media-derived frames.

Start with 4-8 frames on a free T4. If the run succeeds, increase frame count gradually.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path('/content/fpv_vggt_job')
UPLOAD_ZIP = Path('/content/cloud_vggt_job.zip')

def run(command, cwd=None):
    print('+', ' '.join(str(part) for part in command))
    subprocess.check_call([str(part) for part in command], cwd=str(cwd) if cwd else None)

print('Python:', sys.version)
print('Work dir:', WORK)

In [ ]:
import torch

print('Torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. In Colab, select Runtime > Change runtime type > T4 GPU.')

props = torch.cuda.get_device_properties(0)
name = props.name
memory_gib = props.total_memory / (1024 ** 3)
print('CUDA device:', name)
print(f'GPU memory: {memory_gib:.1f} GiB')

if 'T4' not in name.upper():
    print('This notebook is tuned for T4. Other CUDA GPUs can work, but memory behavior may differ.')

run(['nvidia-smi'])

In [ ]:
from google.colab import files

if not UPLOAD_ZIP.exists():
    print('Upload outputs/cloud_vggt_job/cloud_vggt_job.zip from your local workstation.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise RuntimeError('No .zip file uploaded.')
    shutil.copyfile(zip_names[0], UPLOAD_ZIP)

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(UPLOAD_ZIP) as archive:
    archive.extractall(WORK)

candidates = sorted(WORK.rglob('run_vggt_job.py'))
if not candidates:
    raise FileNotFoundError('run_vggt_job.py was not found inside the uploaded package.')
JOB_DIR = candidates[0].parent
manifest_path = JOB_DIR / 'job_manifest.json'
manifest = json.loads(manifest_path.read_text())

print('Job dir:', JOB_DIR)
print('Clip count:', len(manifest.get('clips', [])))
for clip in manifest.get('clips', []):
    print('-', clip['clip_id'], 'frames=', clip.get('frame_count'))

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

WORK = Path("/content/fpv_vggt_job")
DRIVE_ZIP = Path("/content/drive/MyDrive/cloud_vggt_job.zip")

if not DRIVE_ZIP.exists():
    matches = sorted(Path("/content/drive/MyDrive").rglob("cloud_vggt_job.zip"))
    if not matches:
        raise FileNotFoundError("Put cloud_vggt_job.zip in Google Drive, then rerun this cell.")
    DRIVE_ZIP = matches[0]

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP) as archive:
    archive.extractall(WORK)

candidates = sorted(WORK.rglob("run_vggt_job.py"))
if not candidates:
    raise FileNotFoundError("run_vggt_job.py was not found inside the ZIP.")

JOB_DIR = candidates[0].parent
manifest = json.loads((JOB_DIR / "job_manifest.json").read_text())

print("Job dir:", JOB_DIR)
print("Clip count:", len(manifest["clips"]))
for clip in manifest["clips"]:
    print("-", clip["clip_id"], "frames=", clip["frame_count"])

In [ ]:
run([sys.executable, "run_vggt_job.py"], cwd=JOB_DIR)

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

WORK = Path("/content/fpv_vggt_job")
UPLOAD_ZIP = Path("/content/cloud_vggt_job.zip")

if not UPLOAD_ZIP.exists():
    matches = sorted(Path("/content").glob("cloud_vggt_job*.zip"))
    if not matches:
        raise FileNotFoundError(
            "Upload cloud_vggt_job.zip using the left Files panel first."
        )
    shutil.copyfile(matches[0], UPLOAD_ZIP)

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(UPLOAD_ZIP) as archive:
    archive.extractall(WORK)

candidates = sorted(WORK.rglob("run_vggt_job.py"))
if not candidates:
    raise FileNotFoundError("run_vggt_job.py was not found inside the ZIP.")

JOB_DIR = candidates[0].parent
manifest = json.loads((JOB_DIR / "job_manifest.json").read_text())

print("Job dir:", JOB_DIR)
print("Clip count:", len(manifest["clips"]))
for clip in manifest["clips"]:
    print("-", clip["clip_id"], "frames=", clip["frame_count"])

## Run VGGT

This can take time on the first run because VGGT and the model weights are downloaded. If Colab reports CUDA out of memory, return locally and create a smaller package: fewer frames, one clip at a time, and `--resized-long-edge 512`.

In [ ]:
run([sys.executable, 'run_vggt_job.py'], cwd=JOB_DIR)

In [ ]:
summary_path = JOB_DIR / 'cloud_summary.json'
log_path = JOB_DIR / 'cloud_run.log'
bundles_path = JOB_DIR / 'bundles.zip'

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2, sort_keys=True)[:6000])
else:
    raise RuntimeError('cloud_summary.json was not created. Check cloud_run.log if present.')

if not bundles_path.exists():
    raise RuntimeError('bundles.zip was not created. The VGGT job did not finish successfully.')

print('Return files:')
for path in [bundles_path, summary_path, log_path]:
    if path.exists():
        print(path, path.stat().st_size, 'bytes')

In [ ]:
from google.colab import files

for path in [bundles_path, summary_path, log_path]:
    if path.exists():
        files.download(str(path))

## Back On The Local Workstation

Import the returned bundle with:

```bash
fpv vggt import-cloud-job --source <returned-bundles.zip> --output-root data/vggt --report outputs/reviews/cloud_bundle_import.json
```

If that succeeds, continue the normal local review flow. Keep VGGT coordinates labeled as relative and scale ambiguous.